# FaceForensics++ — Kaggle Auto-Training Notebook

**Full automatic training pipeline** — runs on free Kaggle T4/P100 GPU with zero manual intervention.

### What this notebook does automatically:
1. Installs required dependencies
2. Downloads the FaceForensics++ / CIPlab dataset from Kaggle
3. Preprocesses video frames and face crops
4. Trains XceptionNet (ImageNet pretrained) for deepfake detection
5. Evaluates on validation set and exports weights

> **Enable GPU**: Notebook → Settings → Accelerator → GPU T4 x2 (free)

In [ ]:
# ── Cell 1: GPU Check ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install Dependencies ─────────────────────────────────────────────
!pip install -q timm albumentations facenet-pytorch opencv-python-headless tqdm scikit-learn

In [ ]:
# ── Cell 3: Download Dataset from Kaggle ─────────────────────────────────────
# Uses your kaggle.json credentials (automatically available on Kaggle platform)
import os, subprocess

DATASET = 'ciplab/real-and-fake-face-detection'  # Change to your selected dataset
DATA_DIR = '/kaggle/working/data'
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Downloading dataset: {DATASET}')
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', DATASET, '-p', DATA_DIR, '--unzip'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
print('Dataset downloaded!')
!ls -la {DATA_DIR}

In [ ]:
# ── Cell 4: Dataset Exploration & Setup ──────────────────────────────────────
import os, glob, random
from pathlib import Path

DATA_DIR = Path('/kaggle/working/data')

# Find all images (supports CIPlab and FF++ structures)
real_imgs  = list(DATA_DIR.glob('**/real*/**/*.jpg'))  + list(DATA_DIR.glob('**/real*/**/*.png'))
fake_imgs  = list(DATA_DIR.glob('**/fake*/**/*.jpg'))  + list(DATA_DIR.glob('**/fake*/**/*.png'))

# Fallback: flat structure
if not real_imgs:
    all_imgs   = list(DATA_DIR.glob('**/*.jpg')) + list(DATA_DIR.glob('**/*.png'))
    random.shuffle(all_imgs)
    mid        = len(all_imgs) // 2
    real_imgs, fake_imgs = all_imgs[:mid], all_imgs[mid:]

print(f'Real images : {len(real_imgs)}')
print(f'Fake images : {len(fake_imgs)}')
print(f'Total       : {len(real_imgs)+len(fake_imgs)}')

In [ ]:
# ── Cell 5: Dataset Class & Transforms ───────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class DeepfakeDataset(Dataset):
    def __init__(self, real_paths, fake_paths, transform=None):
        self.samples   = [(p, 0) for p in real_paths] + [(p, 1) for p in fake_paths]
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (299, 299))
        if self.transform: img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

# Transforms
train_tf = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])
val_tf = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

# Split 85/15
import random
random.shuffle(real_imgs); random.shuffle(fake_imgs)
cut_r = int(0.85 * len(real_imgs)); cut_f = int(0.85 * len(fake_imgs))
train_ds = DeepfakeDataset(real_imgs[:cut_r], fake_imgs[:cut_f], train_tf)
val_ds   = DeepfakeDataset(real_imgs[cut_r:], fake_imgs[cut_f:], val_tf)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')

In [ ]:
# ── Cell 6: Build XceptionNet Model ──────────────────────────────────────────
import timm, torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

model = timm.create_model('xception', pretrained=True, num_classes=2)
model = model.to(DEVICE)

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f'XceptionNet params: {total:.1f}M')

In [ ]:
# ── Cell 7: Training Loop — Runs Fully Automatically ─────────────────────────
import torch.optim as optim
from tqdm import tqdm

EPOCHS      = 10
LR          = 1e-4
WEIGHT_DECAY= 1e-5
SAVE_PATH   = '/kaggle/working/xceptionnet_faceforensics.pth'

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_acc = 0.0
history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

for epoch in range(1, EPOCHS+1):
    # ── Train ──
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    pbar = tqdm(train_dl, desc=f'Epoch {epoch}/{EPOCHS} [Train]')
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * imgs.size(0)
        preds      = out.argmax(dim=1)
        t_correct += (preds == labels).sum().item()
        t_total   += imgs.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{t_correct/t_total:.4f}')

    # ── Validate ──
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_dl, desc=f'Epoch {epoch}/{EPOCHS} [Val]  '):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out   = model(imgs)
            loss  = criterion(out, labels)
            v_loss += loss.item() * imgs.size(0)
            preds  = out.argmax(dim=1)
            v_correct += (preds == labels).sum().item()
            v_total   += imgs.size(0)

    train_acc = t_correct / max(t_total, 1)
    val_acc   = v_correct / max(v_total, 1)
    train_loss= t_loss    / max(t_total, 1)
    val_loss  = v_loss    / max(v_total, 1)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch}/{EPOCHS} — Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'epoch':epoch,'state_dict':model.state_dict(),'val_acc':val_acc,'val_loss':val_loss}, SAVE_PATH)
        print(f'  ✅ New best model saved (val_acc={val_acc:.4f})')

print(f'\n🎉 Training complete! Best val accuracy: {best_val_acc:.4f}')
print(f'Model saved to: {SAVE_PATH}')

In [ ]:
# ── Cell 8: Plot Training Curves ─────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss', color='#6366f1')
ax1.plot(history['val_loss'],   label='Val Loss',   color='#f43f5e')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.set_xlabel('Epoch')

ax2.plot([a*100 for a in history['train_acc']], label='Train Acc', color='#6366f1')
ax2.plot([a*100 for a in history['val_acc']],   label='Val Acc',   color='#10b981')
ax2.set_title('Accuracy Curve (%)'); ax2.legend(); ax2.set_xlabel('Epoch'); ax2.set_ylabel('%')

plt.suptitle('FaceForensics++ XceptionNet Training', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

In [ ]:
# ── Cell 9: Final Evaluation ──────────────────────────────────────────────────
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

# Load best checkpoint
ckpt = torch.load(SAVE_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.eval()

all_labels, all_probs = [], []
with torch.no_grad():
    for imgs, labels in val_dl:
        imgs = imgs.to(DEVICE)
        out  = torch.softmax(model(imgs), dim=1)
        all_labels.extend(labels.numpy())
        all_probs.extend(out[:,1].cpu().numpy())

preds = [1 if p > 0.5 else 0 for p in all_probs]
auc   = roc_auc_score(all_labels, all_probs)
print(f'AUC Score: {auc:.4f}')
print(classification_report(all_labels, preds, target_names=['Real','Fake']))
print(f'\n✅ Final Best Val Accuracy: {ckpt["val_acc"]:.4f} ({ckpt["val_acc"]*100:.1f}%)')

In [ ]:
# ── Cell 10: List Output Files ────────────────────────────────────────────────
import os
print('Output files ready for download from Kaggle:')
for f in os.listdir('/kaggle/working/'):
    path = f'/kaggle/working/{f}'
    size = os.path.getsize(path) / 1e6
    print(f'  {f:50s}  {size:.1f} MB')